In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [58]:
demand22 = pd.read_excel("../data/raw/ercot_demand/demand2022.xlsx")
demand23 = pd.read_excel("../data/raw/ercot_demand/demand2023.xlsx")
demand24 = pd.read_excel("../data/raw/ercot_demand/demand2024.xlsx")
demand25 = pd.read_excel("../data/raw/ercot_demand/demand2025.xlsx")
demand26 = pd.read_excel("../data/raw/ercot_demand/demand2026.xlsx")

In [45]:
prices22 = pd.read_excel("../data/raw/ercot_prices/prices2022.xlsx", sheet_name=None)
prices23 = pd.read_excel("../data/raw/ercot_prices/prices2023.xlsx", sheet_name=None)
prices24 = pd.read_excel("../data/raw/ercot_prices/prices2024.xlsx", sheet_name=None)
prices25 = pd.read_excel("../data/raw/ercot_prices/prices2025.xlsx", sheet_name=None)
prices26 = pd.read_excel("../data/raw/ercot_prices/prices2026.xlsx", sheet_name=None)

In [71]:
utility = pd.read_excel("../data/raw/eia860/ppe2024/1___Utility_Y2024.xlsx")
plant = pd.read_excel("../data/raw/eia860/ppe2024/2___Plant_Y2024.xlsx")
generator = pd.read_excel("../data/raw/eia860/ppe2024/3_1_Generator_Y2024.xlsx")
wind = pd.read_excel("../data/raw/eia860/ppe2024/3_2_Wind_Y2024.xlsx") 
solar = pd.read_excel("../data/raw/eia860/ppe2024/3_3_Solar_Y2024.xlsx") 
storage = pd.read_excel("../data/raw/eia860/ppe2024/3_4_Energy_Storage_Y2024.xlsx", skiprows=1, sheet_name="Operable")
storage_proposed = pd.read_excel("../data/raw/eia860/ppe2024/3_4_Energy_Storage_Y2024.xlsx", skiprows=1, sheet_name="Proposed")
storage_retired = pd.read_excel("../data/raw/eia860/ppe2024/3_4_Energy_Storage_Y2024.xlsx", skiprows=1, sheet_name="Retired and Canceled")
multifuel = pd.read_excel("../data/raw/eia860/ppe2024/3_5_Multifuel_Y2024.xlsx")
owner = pd.read_excel("../data/raw/eia860/ppe2024/4___Owner_Y2024.xlsx") 
enviroassoc = pd.read_excel("../data/raw/eia860/ppe2024/6_1_EnviroAssoc_Y2024.xlsx") 
enviroequip = pd.read_excel("../data/raw/eia860/ppe2024/6_2_EnviroEquip_Y2024.xlsx") 

In [75]:
storage_tx = storage[storage["State"] == "TX"]
storage_tx

,Utility ID,Utility Name,Plant Code,Plant Name,State,County,Generator ID,Status,Technology,Prime Mover,...,DC Tightly Coupled,Independent,Direct Support of Another Unit,Direct Support Plant ID 1,Direct Support Gen ID 1,Direct Support Plant ID 2,Direct Support Gen ID 2,Direct Support Plant ID 3,Direct Support Gen ID 3,Support T&D Asset
20,55983,Luminant Generation Company LLC,8063,DeCordova Steam Electric Station,TX,Hood,BESS,OP,Batteries,BA,...,NaN,NaN,N,,NaN,,NaN,,NaN,N
35,65411,Duke Energy Renewables Services,56961,Notrees Windpower Hybrid,TX,Ector,BATT,OP,Batteries,BA,...,NaN,NaN,Y,56961,GE,56951,VESTA,,NaN,Y
36,56215,"Big Sky Wind, LLC",56981,Pyron Wind Farm LLC Hybrid,TX,Fisher,PYRBT,OP,Batteries,BA,...,NaN,NaN,Y,56981,1,,NaN,,NaN,Y
37,56215,"Big Sky Wind, LLC",56984,Inadale Wind Farm LLC Hybrid,TX,Nolan,INABT,OP,Batteries,BA,...,NaN,NaN,Y,56984,1,,NaN,,NaN,Y
63,58939,Cameron Wind 1 LLC,59118,Cameron Wind 1 LLC,TX,Cameron,SABAL,OP,Batteries,BA,...,NaN,NaN,N,,NaN,,NaN,,NaN,Y
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
774,67035,Prologis Logistics Services Incorporated BESS,68684,Century BESS,TX,Tarrant,DL805,OP,Batteries,BA,...,NaN,NaN,N,,NaN,,NaN,,NaN,N
775,67035,Prologis Logistics Services Incorporated BESS,68685,Liggett Switch,TX,Tarrant,DL745,OP,Batteries,BA,...,NaN,NaN,N,,NaN,,NaN,,NaN,N
780,67132,Longhorn Storage LLC,68878,Longhorn Storage,TX,Eastland,LONGH,OP,Batteries,BA,...,NaN,NaN,N,,NaN,,NaN,,NaN,N
781,67139,Libra Storage LLC,68895,Libra Storage,TX,Guadalupe,LIBR,OP,Batteries,BA,...,NaN,NaN,N,,NaN,,NaN,,NaN,N


In [54]:
def load_price_workbook(path):
    sheets = pd.read_excel(path, sheet_name=None)
    clean_sheets = []

    for month, df in sheets.items():
        if df is None or df.empty:
            continue

        df = df.dropna(axis=1, how="all")

        if df.empty or df.isna().all().all():
            continue

        df = df.copy()
        df["month"] = month
        clean_sheets.append(df)

    return pd.concat(clean_sheets, ignore_index=True)

prices22 = load_price_workbook("../data/raw/ercot_prices/prices2022.xlsx")
prices23 = load_price_workbook("../data/raw/ercot_prices/prices2023.xlsx")
prices24 = load_price_workbook("../data/raw/ercot_prices/prices2024.xlsx")
prices25 = load_price_workbook("../data/raw/ercot_prices/prices2025.xlsx")
prices26 = load_price_workbook("../data/raw/ercot_prices/prices2026.xlsx")

prices = pd.concat([prices22, prices23, prices24, prices25, prices26], ignore_index=True)

In [61]:
demand = pd.concat([demand22, demand23, demand24, demand25, demand26], ignore_index=True)

In [62]:
demand

,Hour Ending,COAST,EAST,FWEST,NORTH,NCENT,SOUTH,SCENT,WEST,ERCOT
0,01/01/2022 01:00,12054.939199,1302.296674,4161.193625,757.843076,9676.300802,3172.878316,5908.031505,973.455700,38006.938896
1,01/01/2022 02:00,11793.290315,1259.355201,4147.907009,737.236591,9307.126712,3123.318608,5708.512022,959.775908,37036.522365
2,01/01/2022 03:00,11460.841252,1210.287905,4156.412366,725.610502,8920.424552,3003.396233,5463.522829,941.112359,35881.607998
3,01/01/2022 04:00,11244.980243,1179.311517,4149.811722,717.420214,8678.807826,2898.097471,5255.252404,920.373708,35044.055105
4,01/01/2022 05:00,11073.085585,1171.841803,4140.619028,719.178247,8573.370608,2825.100402,5164.172158,918.203309,34585.571140
...,...,...,...,...,...,...,...,...,...,...
36475,02/28/2026 20:00,13384.344352,1615.238171,8064.066485,1653.754523,14161.983096,4674.221302,8539.556822,1552.818615,53645.983365
36476,02/28/2026 21:00,13076.107090,1600.780872,7986.899877,1622.244586,13729.409779,4479.730746,8224.162527,1493.427718,52212.763196
36477,02/28/2026 22:00,12848.024581,1540.855898,7816.771438,1573.028331,13284.799750,4305.154272,7909.851433,1467.112401,50745.598104
36478,02/28/2026 23:00,12498.471987,1455.548588,7671.539886,1498.822125,12686.623820,4166.021672,7481.476314,1425.201471,48883.705863


In [65]:
storage.columns

Index(['2024 Form EIA-860 Data - Schedule 3, 'Energy Storage Data' (Operable Units Only)',
       'Unnamed: 1', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4', 'Unnamed: 5',
       'Unnamed: 6', 'Unnamed: 7', 'Unnamed: 8', 'Unnamed: 9', 'Unnamed: 10',
       'Unnamed: 11', 'Unnamed: 12', 'Unnamed: 13', 'Unnamed: 14',
       'Unnamed: 15', 'Unnamed: 16', 'Unnamed: 17', 'Unnamed: 18',
       'Unnamed: 19', 'Unnamed: 20', 'Unnamed: 21', 'Unnamed: 22',
       'Unnamed: 23', 'Unnamed: 24', 'Unnamed: 25', 'Unnamed: 26',
       'Unnamed: 27', 'Unnamed: 28', 'Unnamed: 29', 'Unnamed: 30',
       'Unnamed: 31', 'Unnamed: 32', 'Unnamed: 33', 'Unnamed: 34',
       'Unnamed: 35', 'Unnamed: 36', 'Unnamed: 37', 'Unnamed: 38',
       'Unnamed: 39', 'Unnamed: 40', 'Unnamed: 41', 'Unnamed: 42',
       'Unnamed: 43', 'Unnamed: 44', 'Unnamed: 45', 'Unnamed: 46',
       'Unnamed: 47', 'Unnamed: 48'],
      dtype='object')